# D4 课外作业：三道 API 题

> 全部基于 **GitHub 公开 API**，不需要 Key，不会花钱。
>
> 三道题难度递增：
>
> | 题 | 练什么 | 新东西 |
> |---|---|---|
> | 作业 1 | 状态码判断 + 剥字段 | 没有，全是今天学的 |
> | 作业 2 | 从 API 数据里**挑几个字段组一个新字典** | 组装字典 |
> | 作业 3 | **返回列表的接口** + 排序 | 数据是列表不是字典 |

**怎么用**：填骨架格 → 运行自测格 → 看到 `通过 ✅` 就过关。

**注意**：GitHub 对未登录的请求有频率限制（每小时 60 次）。做的时候别在一个格子里反复重跑几十次。


---

## 作业 1：查一个仓库有多少 star

**要求**：传入仓库的主人名和仓库名，返回 star 数。**查不到返回 `None`。**

**接口**：`https://api.github.com/repos/{主人}/{仓库名}`

**例子：**

```
get_stars("python", "cpython")     →  77761（数字会随时间变化）
get_stars("不存在的主人", "不存在的仓库")  →  None
```

**骨架：**

```
def get_stars(owner, repo):
    网址 = f"https://api.github.com/repos/{owner}/{repo}"    # ① 拼网址
    响应 = requests.get(网址)                                 # ② 发请求

    # ③ 想一想：状态码不是 200 的时候怎么办？
    #    提示：return None

    数据 = 响应.json()                                        # ④ 转成字典
    return 数据["stargazers_count"]                           # ⑤ 取出 star 数
```

**这五步和今天的 `get_followers` 完全一样**，只是网址和字段名不同。


In [3]:
import requests

def get_stars(owner, repo):
    网址 = f"https://api.github.com/repos/{owner}/{repo}"
    响应 = requests.get(网址)

    # 想一想：状态码不是 200 怎么处理？
    if 响应.status_code != 200:
        return None
    # 提示：return None

    数据 = 响应.json()
    return 数据["stargazers_count"]


# 自测
assert get_stars("python", "cpython") > 50000, "cpython 的 star 数应该很多"
assert get_stars("no-such-owner-xyz", "no-such-repo-xyz") is None, "查不到应该返回 None"
print("作业 1 通过 ✅")

作业 1 通过 ✅


---

## 作业 2：做一张「资料卡」

**要求**：传入用户名，返回一个**你自己组装的字典**，只挑三个字段。查不到返回 `None`。

**接口**：`https://api.github.com/users/{用户名}`

**例子：**

```
get_profile("octocat")
→  {"name": "The Octocat", "followers": 24586, "repos": 8}
```

**注意返回的形状不同了**：作业 1 返回一个**数字**，这次返回一个**字典**。

字典的键名由**你**决定（用 `name` / `followers` / `repos` 这三个），值从 API 数据里取。

**API 给的字段名是这样的**（注意大小写和下划线，必须一模一样）：

| 你想返回的键 | API 里的字段名 |
|---|---|
| `"name"` | `数据["name"]` |
| `"followers"` | `数据["followers"]` |
| `"repos"` | `数据["public_repos"]` ← **注意是 `public_repos`，不是 `repos`** |

**骨架：**

```
def get_profile(username):
    网址 = f"https://api.github.com/users/{username}"
    响应 = requests.get(网址)

    # ① 想一想：状态码不是 200 就 return None

    数据 = 响应.json()

    # ② 想一想：怎么组装一个新字典并返回？
    #    提示：return {"name": 数据["name"], "followers": ..., "repos": ...}
```

**为什么这题值得做**：真实项目里，API 返回的字段往往有一大堆（几十个），你只挑要用的几个重新组一个干净的字典。**这个动作在 RAG 和 Agent 里到处都是。**


In [5]:
def get_profile(username):
    网址 = f"https://api.github.com/users/{username}"
    响应 = requests.get(网址)

    # ① 想一想：状态码不是 200 怎么办？
    if 响应.status_code !=200:
        return None
    数据 = 响应.json()
    return {"name": 数据["name"], "followers": 数据["followers"], "repos": 数据["public_repos"]}

    # ② 想一想：怎么组装一个新字典返回？
    # 提示：return {"name": ..., "followers": ..., "repos": ...}


# 自测
资料 = get_profile("octocat")
assert 资料["name"] == "The Octocat"
assert 资料["followers"] > 20000
assert 资料["repos"] > 0
assert len(资料) == 3, "返回值应该只有 3 个键"
assert get_profile("no-such-user-xyz-9527") is None
print("作业 2 通过 ✅")

作业 2 通过 ✅


---

## 作业 3（挑战）：列出 star 最多的前 n 个仓库

**这一题有真正的新东西：接口返回的是一个【列表】，不是字典。**

**接口**：`https://api.github.com/users/{用户名}/repos`（注意结尾多了 `/repos`）

**要求**：返回 star 最多的前 `n` 个仓库的**名字列表**。查不到返回 `[]`。

**例子：**

```
top_repos("octocat", 3)  →  ["Spoon-Knife", "Hello-World", "octocat.github.io"]
```

### 先看清返回的东西长什么样

这个接口返回的不是一个字典，而是**一个列表，里面装着好几个字典**：

```
[
    {"name": "boysenberry-repo-1", "stargazers_count": 483,  "language": None, ...},
    {"name": "git-consortium",     "stargazers_count": 619,  "language": None, ...},
    {"name": "Spoon-Knife",        "stargazers_count": 14112, "language": "HTML", ...},
    ...
]
```

**怎么读这个结构**：

| 写法 | 拿到的是什么 |
|---|---|
| `数据` | 一个列表 |
| `数据[0]` | 列表第 0 个元素 —— **一个字典** |
| `数据[0]["name"]` | 那个字典里 `name` 的值 |

**这和你昨天剥 `choices[0]["message"]` 是同一个动作** —— 只是这次外面那层是列表。

### 骨架

```
def top_repos(username, n):
    网址 = f"https://api.github.com/users/{username}/repos"
    响应 = requests.get(网址)

    # ① 状态码不是 200 → return []

    数据 = 响应.json()          # 这次 数据 是一个【列表】

    # ② 想一想：怎么按 star 从大到小排序？
    #    提示：昨天学过 —— sorted(列表, key=lambda x: x["字段"], reverse=True)
    #    这里的「字段」就是 stargazers_count

    排序后 = ???

    # ③ 想一想：怎么把前 n 个的 name 拿出来，组成一个新列表？
    #    提示：建一个空列表，for 循环遍历 排序后[:n]，把 repo["name"] append 进去

    return ???
```

**第 ② 步用到的 `sorted` + `lambda`，就是你昨天在关键词排行榜里写的那一行。** 只不过昨天排序的是「词和次数」，今天排序的是「仓库」。


In [1]:
# ## 修正后的完整版本（三处都改了）

# def top_repos(username, n):
#     网址 = f"https://api.github.com/users/{username}/repos"
#     响应 = requests.get(网址)
#     if 响应.status_code != 200:
#         return []

#     数据 = 响应.json()

#     # 按 star 数从大到小排序（不是 language）
#     排序后 = sorted(数据, key=lambda repo: repo["stargazers_count"], reverse=True)

#     结果 = []
#     for repo in 排序后[:n]:
#         结果.append(repo["name"])          # 取仓库名，不是 owner
#     return 结果
第一步 装库request  import request
第二步定义函数 get_language(owner,repo) def get_language(owner,n):
第三步 填网址    网址 = f"xxxxxxxxxxxx/owner/n"
第四步 获取网址  响应 = request.get(网址)
第五步判断网址状态 if 响应.status_code
第六步转化成json文件 数据 = 响应.json
第七步排序 sorted(数据，key=lambda repo: repo[""],reverse=Ture)
第八步将答案导入结果列表for repo in sorted(数据，key=lambda repo: repo[""],reverse=Ture)
                        结果.append(repo["name"])
第九步返回结果

---

## 为什么作业 3 最重要

因为 **「拿到一个列表，按某个字段排序，取前几个」这个动作，就是 RAG 检索的核心步骤。**

对比一下：

| 今天 | 两周后的 RAG |
|---|---|
| 数据 = 一堆仓库（列表） | 数据 = 一堆文档片段（列表） |
| 按 `stargazers_count` 排序 | 按 **向量相似度** 排序 |
| 取前 `n` 个的 `name` | 取前 `n` 段的**文字** |

**骨架一模一样。** 你两周后要写的检索代码，和今天这道题的结构是同一个。

区别只有一个：今天用「star 数」当分数，RAG 里用「相似度」当分数。**两者都是「给每个候选打个分，然后排序取最高的几个」。**

---

## 卡住了怎么办

| 报错 | 什么意思 | 怎么办 |
|---|---|---|
| `KeyError: 'stargazers_count'` | 这个字段不存在 | 把 `数据` 或 `数据[0]` 打印出来，看真实字段名 |
| `TypeError: list indices must be integers` | 你把列表当成字典用了 | 列表要用**数字**下标 `[0]`，字典才能用文字键 `["name"]` |
| `NameError: name 'request' is not defined` | 少写了 s | 是 `requests` |
| `AssertionError` | 某条用例不符合预期 | 看是第几条，把那组输入单独跑一遍 |
| 返回 `None` 但你写了 return | 代码改了没重跑 | Shift+Enter 重新运行那一格 |

**万能排查法还是那一句：**

```
print(响应.status_code)
print(响应.json())
```

**把中间结果打出来看。** 你现在应该已经形成这个条件反射了。

---

## 做完之后

三道题全过之后，把这一格加到最后运行一遍，看看你的成果：

```
print(get_stars("python", "cpython"))
print(get_profile("torvalds"))
print(top_repos("torvalds", 3))
```

**如果这三行都能打印出合理的结果，说明你已经能熟练调用任何 REST API 了。** 这是后面所有项目的地基。


In [15]:
print(get_stars("python", "cpython"))
print(get_profile("torvalds"))
print(top_repos("torvalds", 3))


77762
{'name': 'Linus Torvalds', 'followers': 327831, 'repos': 12}
['linux', 'AudioNoise', 'GuitarPedal']
